In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from lens.raytrace import trace, cross, best_spot
from lens.design import R2_for, recolour
from lens.detect import chromatic

best_form = [(0, 29.5, 1.5), (6, R2_for(29.5, 6, 50), 1.0)]
f5 = np.linspace(-5, 5, 1001)
n_blue, n_green, n_red = 1.50558, 1.5, 1.49752

print(recolour(best_form, n_blue))

[(0, 29.5, 1.50558), (6, -152.77777777777774, 1.0)]


**one lens, three colours.** glass bends blue light slightly more than red (its refractive index depends on colour). recolour makes the same lens with a different index for its glass, so each colour can be traced separately. here: blue n = 1.50558, green n = 1.5, red n = 1.49752, our n = 1.5 with the colour spread of a real, common glass (N-BK7).

In [2]:
for name, n in [("blue", n_blue), ("green", n_green), ("red", n_red)]:
    lens = recolour(best_form, n)
    print(name, cross(0.01, lens), best_spot(lens, 0.0, f5))

blue 52.07467829093321 (np.float64(51.75), np.float64(0.008091266990639806))
green 52.610167419773845 (np.float64(52.3), np.float64(0.00801864764938113))
red 52.852068251659354 (np.float64(52.55), np.float64(0.008030636087479514))


**each colour focuses in a different place.** near-axis rays cross at 52.07 mm (blue), 52.61 mm (green) and 52.85 mm (red), a 0.78 mm spread with blue closest to the lens. each colour on its own is just as sharp (about 0.008 mm at f/5); the problem is only that they don't focus together.

In [3]:
print(chromatic(best_form, n_blue, n_red))

-0.777389960726147


**chromatic aberration detector.** traces a near-axis ray in blue and in red and compares where they cross the axis. for the best-form lens, blue focuses 0.78 mm closer to the lens than red: chromatic aberration of −0.78 mm.

In [4]:
best_form = [(0, 29.5, 1.5), (6, R2_for(29.5, 6, 50), 1.0)]
plano_convex = [(0, 25, 1.5), (5, np.inf, 1.0)]
winner = [(0, 100, 1.5), (6, R2_for(100, 6, 50), 1.0)]
print(chromatic(best_form, n_blue, n_red))
print(chromatic(plano_convex, n_blue, n_red))
print(chromatic(winner, n_blue, n_red))

-0.777389960726147
-0.7832033451851714
-0.7877210774560979


chromatic aberration of three lenses (blue vs red, near-axis):

| lens | chromatic (mm) |
|---|---|
| plano-convex | −0.783 |
| best-form | −0.777 |
| winner | −0.788 |

all three are almost identical (within 0.01 mm), even though they were wildly different on spherical aberration and field curvature. the lens shape barely affects colour: it depends on the glass and the focal length, and all three are the same glass with f = 50 mm.

In [5]:
V = (n_green - 1) / (n_blue - n_red)
print(V)
print(50/V)

62.034739454094634
0.8059999999999956


the abbe number V = (n_green − 1) / (n_blue − n_red) measures how much a glass spreads colours: a big V means little spread. for our glass V = 0.5 / 0.00806 = 62.0. for a single thin lens the colour spread is roughly f / V = 50 / 62.0 = 0.81 mm, close to the detector's 0.78 mm (the lens isn't thin, so it's not exact). since f is fixed at 50 mm, the only way to shrink colour in a single lens is a glass with a bigger V.

In [6]:
zb, yb, ub = trace(f5, recolour(best_form, n_blue))
zg, yg, ug = trace(f5, recolour(best_form, n_green))
zr, yr, ur = trace(f5, recolour(best_form, n_red))

z_last = np.concatenate([zb, zg, zr])
y = np.concatenate([yb, yg, yr])
u = np.concatenate([ub, ug, ur])

sensors = np.linspace(45, 56, 221)
spots = []
for z in sensors:
    y_sensor = y + np.tan(u) * (z - z_last)
    spots.append(np.std(y_sensor))
spots = np.array(spots)
best = np.argmin(spots)
print(sensors[best], spots[best])

52.2 0.020610168809890788


white light: the best-form lens traced in blue, green and red at f/5 (1001 rays each), all 3003 rays combined into one bundle, and the sensor scanned from 45 to 56 mm.

| light | best sensor | spot |
|---|---|---|
| green only | 52.3 mm | 0.0080 mm |
| white (blue + green + red) | 52.2 mm | 0.0206 mm |

white light is about 2.6x blurrier than any single colour. each colour on its own is just as sharp, but they focus in different places, and a camera records all of them at once. the best compromise sensor should sit between the blue and red foci.

In [7]:
ch = chromatic(best_form, n_blue, n_red)
blur = np.abs(ch) * (5 / 50)
print(blur)

0.0777389960726147


chromatic aberration as blur at f/5 (the same conversion as diagnose: × a / f = 5 / 50 = 0.1): 0.78 mm × 0.1 = 0.078 mm.

| aberration (best-form lens) | blur |
|---|---|
| field curvature | 0.255 mm |
| chromatic | 0.078 mm |
| spherical | 0.052 mm |
| coma | 0.016 mm |

colour is the second-biggest problem, ahead of spherical aberration, which the best-form shape was optimized to reduce. and since bending the lens can't change it, it has to be fixed with the glass!